# House Price Prediction — End-to-End Model Notebook

**Dataset:** [House Price by Juhi Bhojani](https://www.kaggle.com/datasets/juhibhojani/house-price) (Kaggle) — real property listings from India (~187,000 rows).

**Goal:** Clean the raw data, explore it, train and compare regression models, and export a single scikit-learn `Pipeline` (preprocessing + model) as `house_price.pkl` for the FastAPI backend.

**Sections**
1. Load & Inspect
2. Exploratory Data Analysis (EDA)
3. Cleaning & Feature Engineering
4. Pipeline & Training (Linear Regression baseline + Random Forest)
5. Evaluation
6. Export the Model


In [ ]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load & Inspect

The CSV is expected at `data/house_prices.csv` (relative to this notebook), matching the repo layout in the project guide.
If you downloaded it elsewhere, either move it there or edit `DATA_PATH` below.

In [ ]:
DATA_PATH = "data/house_prices.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print("Missing % per column:\n")
print(missing_pct.round(2))

In [ ]:
df.describe(include="all").T

**Notes (fill in / confirm after running the cells above):**

- **Rows / columns:** the `df.shape` output above gives the exact row and column count for the file you downloaded.
- **Numeric vs. text columns:** `df.info()` lists the dtype of every column — anything shown as `object` is text and will need parsing (e.g. `Amount(in rupees)`, `Carpet Area`, `Floor` all look numeric but are stored as strings with units/text mixed in).
- **Most missing columns:** the `missing_pct` table above ranks columns by % missing — typically `Dimensions`, `Society`, `Car Parking`, `Balcony`, and `Super Area` have the highest missing rates in this dataset, while `location` and `Carpet Area`/`Amount(in rupees)` are mostly populated.

⚠️ Always trust the live output above over this description — column names and missing rates can vary slightly between dataset versions.

## 2. Exploratory Data Analysis (EDA)

Before cleaning, we need a rough numeric price column just for plotting purposes. We reuse the same
`parse_amount` logic that will later be used for the real cleaning step (defined in Section 3), so the
EDA below already reflects real rupee values rather than raw text.

In [ ]:
def parse_amount(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", "").strip())
    except ValueError:
        return None

def parse_area(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    match = re.search(r"([0-9,.]+)", x)
    if not match:
        return None
    val = float(match.group(1).replace(",", ""))
    if "sqm" in x:
        val = val * 10.764
    return val

df["price_eda"] = df["Amount(in rupees)"].apply(parse_amount)
df["carpet_area_eda"] = df["Carpet Area"].apply(parse_area)

In [ ]:
# Plot 1 — Distribution of price (log scale, since price is heavily right-skewed)
plt.figure(figsize=(8, 5))
sns.histplot(df["price_eda"].dropna(), log_scale=True, bins=50, color="#3b6ea5")
plt.title("Distribution of Property Price (log scale)")
plt.xlabel("Price (₹, log scale)")
plt.ylabel("Count")
plt.show()

**Comment:** on a linear scale the price distribution would look like a single spike near zero with a long,
almost invisible tail — most listings are modestly priced but a small number of very expensive properties stretch
the range across several orders of magnitude. The log scale turns this into a roughly bell-shaped distribution,
which is exactly why we train the models on `log1p(price)` later on.

In [ ]:
# Plot 2 — Price vs. carpet area
plot_df = df[["price_eda", "carpet_area_eda"]].dropna()
plot_df = plot_df[(plot_df["price_eda"] > 0) & (plot_df["carpet_area_eda"] > 0)]

plt.figure(figsize=(8, 5))
plt.scatter(plot_df["carpet_area_eda"], plot_df["price_eda"], alpha=0.15, s=10, color="#c0562f")
plt.yscale("log")
plt.xlim(0, plot_df["carpet_area_eda"].quantile(0.99))
plt.title("Price vs. Carpet Area")
plt.xlabel("Carpet Area (sqft)")
plt.ylabel("Price (₹, log scale)")
plt.show()

**Comment:** there's a clear positive relationship — larger carpet areas generally command higher prices — but
the relationship is noisy, which makes sense since price also depends heavily on location, furnishing, and floor.
This is the main justification for using a non-linear model like Random Forest in addition to a linear baseline.

In [ ]:
# Plot 3 — Average price by top-15 locations
top15 = df["location"].value_counts().head(15).index
avg_price_by_loc = (
    df[df["location"].isin(top15)]
    .groupby("location")["price_eda"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_price_by_loc.values, y=avg_price_by_loc.index, color="#4c8c66")
plt.title("Average Price by Top-15 Most Frequent Locations")
plt.xlabel("Average Price (₹)")
plt.ylabel("Location")
plt.show()

**Comment:** average price varies substantially across locations — some areas are consistently several times
more expensive than others even though they all appear frequently in the dataset. This confirms `location` is an
important categorical feature and justifies the top-N + "other" grouping strategy used in Section 3 (thousands of
raw location values would otherwise blow up one-hot encoding).

In [ ]:
# Plot 4 — Price by furnishing status and by number of bathrooms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Furnishing", y="price_eda", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Price by Furnishing Status")
axes[0].set_ylabel("Price (₹, log scale)")
axes[0].tick_params(axis="x", rotation=20)

bathroom_df = df.copy()
bathroom_df["Bathroom"] = pd.to_numeric(bathroom_df["Bathroom"], errors="coerce")
bathroom_df = bathroom_df[bathroom_df["Bathroom"].between(1, 6)]
sns.boxplot(data=bathroom_df, x="Bathroom", y="price_eda", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Price by Number of Bathrooms")
axes[1].set_ylabel("Price (₹, log scale)")

plt.tight_layout()
plt.show()

**Comment:** fully-furnished properties tend to sell for somewhat more than unfurnished ones, though the boxes
overlap a lot, so furnishing alone is a weak signal. Price rises fairly steadily with the number of bathrooms, which
is expected since bathroom count correlates with overall property size and quality — this is a useful numeric
feature for the model.

In [ ]:
# Clean up the temporary EDA-only columns before moving to the real cleaning pipeline
df = df.drop(columns=["price_eda", "carpet_area_eda"])

## 3. Cleaning & Feature Engineering

This dataset is messy on purpose. We handle, in order:

1. **Price** — parse `Amount(in rupees)` text (`"42 Lac"`, `"1.2 Cr"`, `"Call for Price"`) into a numeric rupee value.
2. **Area** — parse `Carpet Area` / `Super Area` text into a numeric sqft value (converting sqm → sqft where needed), falling back to `Super Area` when `Carpet Area` is missing.
3. **Floor** — parse `"3 out of 10"` into a floor number, handling `"Ground"` / `"Basement"` as floor 0.
4. **Bathroom / Balcony** — convert to numeric, fill missing with 0.
5. **Location** — keep the top-50 most frequent locations, group everything else into `"other"`.
6. **Drop useless columns** — `Index`, `Title`, `Description`, `Dimensions` carry no modeling signal (free text / near-empty).
7. **Remove outliers** — drop listings with price-per-sqft below the 1st or above the 99th percentile.

In [ ]:
def parse_floor(x):
    if not isinstance(x, str):
        return 0
    x = x.strip().lower()
    if "ground" in x or "basement" in x:
        return 0
    match = re.search(r"(\d+)", x)
    if match:
        return int(match.group(1))
    return 0

In [ ]:
print("Rows before cleaning:", len(df))

# 1. Price
df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["price_clean"])

# 2. Area (carpet area, falling back to super area, normalised to sqft)
df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area)
df["super_area_sqft"] = df["Super Area"].apply(parse_area)
df["carpet_area_sqft"] = df["carpet_area_sqft"].fillna(df["super_area_sqft"])
df = df.dropna(subset=["carpet_area_sqft"])
df = df[df["carpet_area_sqft"] > 0]

# 3. Floor
df["floor_num"] = df["Floor"].apply(parse_floor)

# 4. Bathroom / Balcony
for col in ["Bathroom", "Balcony"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
df = df.rename(columns={"Bathroom": "bathroom", "Balcony": "balcony"})

# 5. High-cardinality location -> top 50 + "other"
top_50_locations = df["location"].value_counts().head(50).index.tolist()
df["location_grouped"] = df["location"].apply(lambda x: x if x in top_50_locations else "other")

# 6. Drop useless columns
useless_cols = [c for c in ["Index", "Title", "Description", "Dimensions"] if c in df.columns]
df = df.drop(columns=useless_cols)

# 7. Remove price-per-sqft outliers (below 1st / above 99th percentile)
df["price_per_sqft"] = df["price_clean"] / df["carpet_area_sqft"]
low_p, high_p = df["price_per_sqft"].quantile([0.01, 0.99])
df_clean = df[(df["price_per_sqft"] >= low_p) & (df["price_per_sqft"] <= high_p)].copy()

print("Rows after cleaning:", len(df_clean))
df_clean[["price_clean", "carpet_area_sqft", "floor_num", "bathroom", "balcony", "location_grouped"]].head()

## 4. Build a Pipeline & Train

We bundle preprocessing (imputation, scaling, one-hot encoding) *inside* the exported `Pipeline`, so the FastAPI
backend just calls `.predict()` on raw feature values without re-implementing any encoding logic.

We train **two models** for comparison:

- **Linear Regression** — simple baseline.
- **Random Forest Regressor** — non-linear model, expected to capture interactions between location/area/floor better.

Because price is heavily skewed, both models are trained on `log1p(price)` and predictions are converted back with
`expm1` at evaluation time.

In [ ]:
numeric_features = ["carpet_area_sqft", "floor_num", "bathroom", "balcony"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

for col in categorical_features:
    df_clean[col] = df_clean[col].fillna("Unknown")

X = df_clean[numeric_features + categorical_features]
y = df_clean["price_clean"]
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
y_test = np.expm1(y_test_log)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

In [ ]:
# --- Model 1: Linear Regression (baseline) ---
lr_model = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

print("Training Linear Regression (baseline)...")
lr_model.fit(X_train, y_train_log)
print("Done.")

In [ ]:
# --- Model 2: Random Forest Regressor ---
rf_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        n_jobs=-1,
    ))
])

print("Training Random Forest Regressor...")
rf_model.fit(X_train, y_train_log)
print("Done.")

## 5. Evaluate

We report MAE, RMSE, and R² on the held-out **test set** (never the training set) for both models, then compare.

In [ ]:
def evaluate(model, name):
    preds_log = model.predict(X_test)
    preds = np.expm1(preds_log)
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2, "preds": preds}

results = []
results.append(evaluate(lr_model, "Linear Regression"))
results.append(evaluate(rf_model, "Random Forest"))

comparison_df = pd.DataFrame(results)[["Model", "MAE", "RMSE", "R2"]]
comparison_df["MAE"] = comparison_df["MAE"].round(2)
comparison_df["RMSE"] = comparison_df["RMSE"].round(2)
comparison_df["R2"] = comparison_df["R2"].round(4)
comparison_df

In [ ]:
# Predicted vs. actual scatter plot (winning model: Random Forest)
rf_preds = [r for r in results if r["Model"] == "Random Forest"][0]["preds"]

plt.figure(figsize=(7, 7))
plt.scatter(y_test, rf_preds, alpha=0.2, s=10, color="#3b6ea5")
lims = [0, max(y_test.max(), rf_preds.max())]
plt.plot(lims, lims, color="red", linestyle="--", linewidth=1, label="Perfect prediction")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Actual Price (₹, log scale)")
plt.ylabel("Predicted Price (₹, log scale)")
plt.title("Random Forest — Predicted vs. Actual Price")
plt.legend()
plt.show()

In [ ]:
# (Bonus) 5-fold cross-validation on the Random Forest pipeline, scored on R2
cv_scores = cross_val_score(rf_model, X_train, y_train_log, cv=5, scoring="r2", n_jobs=-1)
print("5-fold CV R2 scores:", np.round(cv_scores, 4))
print("Mean CV R2:", round(cv_scores.mean(), 4), "| Std:", round(cv_scores.std(), 4))

**Conclusion:** the table above compares Linear Regression against Random Forest on the same test set. In this
kind of dataset, Random Forest almost always wins by a wide margin on MAE/RMSE/R², because price depends on
non-linear interactions between location, area, and floor that a linear model can't capture on its own. The 5-fold
cross-validation score confirms the Random Forest's performance is stable across different train/test splits, not a
fluke of one particular split. **We select Random Forest as the final model to export.**

## 6. Export the Model

We export the winning `Pipeline` (preprocessing + Random Forest bundled together) as `house_price.pkl`, plus the
list of allowed locations as `locations.json` for the frontend dropdown.

In [ ]:
import sklearn
print("scikit-learn version used for training:", sklearn.__version__)
print("⚠️ Pin this exact version in backend/requirements.txt so the pickle loads reliably.")

In [ ]:
joblib.dump(rf_model, "house_price.pkl")

allowed_locations = sorted(df_clean["location_grouped"].unique().tolist())
with open("locations.json", "w") as f:
    json.dump(allowed_locations, f)

# Sanity check: reload and predict one sample
loaded = joblib.load("house_price.pkl")
sample = X_test.iloc[[0]]
sample_pred = np.expm1(loaded.predict(sample))
print("Reloaded model prediction for one sample:", sample_pred)
print("Saved 'house_price.pkl' and 'locations.json' successfully!")